# Mobile Money Agent Network Data Clean-Up

## Data Quality Audit and Cleaning Report

This notebook audits, cleans, validates, merges and summarises the mobile-money agent transaction data. The analysis follows the six dimensions of data quality: completeness, validity, accuracy, consistency, uniqueness and timeliness.

In [ ]:
import pandas as pd
import numpy as np

from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")


print("Transactions shape:", txns.shape)
print("Agents shape:", agents.shape)

display(txns.head())
display(agents.head())

Transactions shape: (612, 5)
Agents shape: (20, 3)


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


## Task 10: Data Quality Audit

I audited the transaction data using the six dimensions of data quality. The audit checks for missing values, duplicate transaction IDs, inconsistent transaction types, invalid amount values, inconsistent date formats and transactions linked to unknown agents.

In [2]:
# Basic audit counts

duplicate_txn_ids = txns.duplicated(subset=["txn_id"]).sum()

missing_amounts = txns["amount"].isna().sum()

valid_txn_types = ["Deposit", "Withdrawal"]
standardised_txn_type = (
    txns["txn_type"]
    .astype("string")
    .str.strip()
    .str.title()
)

invalid_txn_types = (~standardised_txn_type.isin(valid_txn_types)).sum()

amount_numeric = pd.to_numeric(
    txns["amount"],
    errors="coerce"
)

invalid_amounts = amount_numeric.isna().sum()

slash_dates = (
    txns["txn_date"]
    .astype("string")
    .str.contains("/", regex=False)
    .sum()
)

unknown_agents = (
    ~txns["agent_id"].isin(agents["agent_id"])
).sum()

audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": duplicate_txn_ids
    },
    {
        "issue": "Missing amount values",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": missing_amounts
    },
    {
        "issue": "Inconsistent transaction type labels",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": invalid_txn_types
    },
    {
        "issue": "Amounts that cannot be converted to numeric",
        "dimension": "Validity",
        "column": "amount",
        "rows_affected": invalid_amounts
    },
    {
        "issue": "Dates using slash separators",
        "dimension": "Consistency",
        "column": "txn_date",
        "rows_affected": slash_dates
    },
    {
        "issue": "Transactions linked to unknown agents",
        "dimension": "Accuracy",
        "column": "agent_id",
        "rows_affected": unknown_agents
    }
])

display(audit)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing amount values,Completeness,amount,22
2,Inconsistent transaction type labels,Consistency,txn_type,0
3,Amounts that cannot be converted to numeric,Validity,amount,22
4,Dates using slash separators,Consistency,txn_date,5
5,Transactions linked to unknown agents,Accuracy,agent_id,8


In [3]:
# Corrected Task 10 audit table
# This cell replaces the earlier audit result.

duplicate_txn_ids = txns.duplicated(subset=["txn_id"]).sum()

missing_amounts = txns["amount"].isna().sum()

# Count raw transaction-type labels that are not already exactly clean
invalid_txn_types = (
    ~txns["txn_type"].isin(["Deposit", "Withdrawal"])
).sum()

amount_numeric = pd.to_numeric(
    txns["amount"],
    errors="coerce"
)

invalid_amounts = amount_numeric.isna().sum()

slash_dates = (
    txns["txn_date"]
    .astype("string")
    .str.contains("/", regex=False)
    .sum()
)

unknown_agents = (
    ~txns["agent_id"].isin(agents["agent_id"])
).sum()

audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows_affected": duplicate_txn_ids
    },
    {
        "issue": "Missing amount values",
        "dimension": "Completeness",
        "column": "amount",
        "rows_affected": missing_amounts
    },
    {
        "issue": "Inconsistent transaction type labels",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows_affected": invalid_txn_types
    },
    {
        "issue": "Amounts that cannot be converted to numeric",
        "dimension": "Validity",
        "column": "amount",
        "rows_affected": invalid_amounts
    },
    {
        "issue": "Dates using slash separators",
        "dimension": "Consistency",
        "column": "txn_date",
        "rows_affected": slash_dates
    },
    {
        "issue": "Transactions linked to unknown agents",
        "dimension": "Accuracy",
        "column": "agent_id",
        "rows_affected": unknown_agents
    }
])

display(audit)

,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing amount values,Completeness,amount,22
2,Inconsistent transaction type labels,Consistency,txn_type,384
3,Amounts that cannot be converted to numeric,Validity,amount,22
4,Dates using slash separators,Consistency,txn_date,5
5,Transactions linked to unknown agents,Accuracy,agent_id,8


## Tasks 11–16: Clean, Validate and Merge Agent Transactions

I applied the reusable `clean_agent_data()` function from `cleaning_pipeline.py`. The function removes duplicate transaction IDs, standardises transaction types, converts amounts and dates to the correct types, imputes missing or invalid amounts, flags IQR outliers, and performs a left join with the agent register so valid transactions are not lost.

In [4]:
clean_df, cleaning_log = clean_agent_data(txns, agents)

print("CLEANING LOG")
for item in cleaning_log:
    print("-", item)

print("\nRAW ROWS:", len(txns))
print("CLEAN ROWS:", len(clean_df))

print("\nDUPLICATE TXN IDs:")
print(clean_df["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES:")
print(clean_df["txn_type"].value_counts(dropna=False))

print("\nMISSING VALUES:")
print(clean_df.isna().sum())

print("\nUNKNOWN AGENT TRANSACTIONS:")
print((clean_df["county"] == "Unknown").sum())

print("\nDATA TYPES:")
print(clean_df.dtypes)

display(clean_df.head())

CLEANING LOG
- Removed 12 duplicate transaction IDs.
- Standardised transaction type to Deposit and Withdrawal; 0 invalid transaction types remained.
- Imputed 20 missing or invalid amounts using the median amount of KES 1776.50.
- Converted transaction dates to datetime; 0 invalid dates remained.
- Flagged 53 amount outliers using the IQR method.
- Found 7 transactions with unknown agents.
- Row count changed from 612 raw rows to 600 cleaned rows.

RAW ROWS: 612
CLEAN ROWS: 600

DUPLICATE TXN IDs:
0

TRANSACTION TYPES:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

MISSING VALUES:
txn_id            0
agent_id          0
txn_type          0
amount            0
txn_date          0
amount_imputed    0
is_outlier        0
county            0
onboarded         7
dtype: int64

UNKNOWN AGENT TRANSACTIONS:
7

DATA TYPES:
txn_id                       str
agent_id                     str
txn_type                     str
amount                   float64
txn_date         

,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01


### Tasks 11–16 Findings

The raw transaction file contained 612 rows, including 12 duplicate transaction IDs. After removing those duplicates, 600 unique transactions remained. The transaction type values were standardised into exactly two categories, Deposit and Withdrawal, resulting in 373 Deposit transactions and 227 Withdrawal transactions.

After duplicate removal, 20 transaction amounts were missing or invalid. These values were imputed using the median valid transaction amount of KES 1,776.50, and the `amount_imputed` flag records which transactions were affected. Transaction dates were successfully converted to datetime format with no invalid dates remaining.

The IQR method flagged 53 transaction amounts as possible outliers. These values were kept in the dataset and flagged for review instead of being deleted automatically because unusually large transactions may still be genuine business activity.

A left join was used to merge transactions with the agent register so that no valid transaction would be lost. Seven cleaned transactions referenced agent IDs that were not found in the agent register, so their county was labelled `Unknown`. This keeps the transactions available for analysis while clearly identifying the agent-reference data-quality problem.

In [5]:
# Investigate high-value transactions and the injected KES 250,000 values

high_value_250k = clean_df[
    clean_df["amount"] == 250000
]

print("KES 250,000 transactions:", len(high_value_250k))

display(
    high_value_250k[
        [
            "txn_id",
            "agent_id",
            "txn_type",
            "amount",
            "txn_date",
            "county",
            "is_outlier"
        ]
    ]
)

print("\nAll IQR outliers:", clean_df["is_outlier"].sum())

KES 250,000 transactions: 5


,txn_id,agent_id,txn_type,amount,txn_date,county,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,Kisumu,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,Nairobi,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,Mombasa,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,Nakuru,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,Kiambu,True



All IQR outliers: 53


### Task 15: Outlier Investigation and Recommendation

Five transactions had an amount of exactly KES 250,000, and all five were flagged as outliers by the IQR method. These transactions occurred across different agents, counties, dates and transaction types, so they should not be deleted automatically just because they are unusually large. The business should verify them against source records or transaction logs before deciding whether they are valid. If confirmed, they should remain in the dataset as genuine high-value transactions; if they are data-entry or system errors, they should be corrected or excluded with the reason documented.

## Task 17: County and Weekly Transaction Summary

I summarised cleaned transactions by county and transaction type, then created a weekly pivot table of transaction value by county.

In [6]:
# County summary split by transaction type
county_summary = (
    clean_df.groupby(["county", "txn_type"])
    .agg(
        total_value=("amount", "sum"),
        transactions=("txn_id", "count"),
        average_value=("amount", "mean")
    )
    .reset_index()
    .sort_values(["county", "txn_type"])
)

display(county_summary.round(2))

# Create week column
clean_df["week"] = clean_df["txn_date"].dt.to_period("W").astype(str)

# Weekly transaction value by county
weekly_pivot = pd.pivot_table(
    clean_df,
    index="county",
    columns="week",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot.round(2))

,county,txn_type,total_value,transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.58
1,Kiambu,Withdrawal,77050.5,27,2853.72
2,Kisumu,Deposit,436420.5,74,5897.57
3,Kisumu,Withdrawal,76374.5,37,2064.18
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.49
6,Nairobi,Deposit,543590.0,118,4606.69
7,Nairobi,Withdrawal,162036.0,62,2613.48
8,Nakuru,Deposit,184890.0,60,3081.50
9,Nakuru,Withdrawal,328594.5,39,8425.50


week,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05,2026-04-06/2026-04-12,2026-04-13/2026-04-19,2026-04-20/2026-04-26,2026-04-27/2026-05-03
county,,,,,,,,,,
Kiambu,0.0,18317.0,267771.5,22054.0,11504.0,22609.0,15030.0,12265.0,24507.0,17730.0
Kisumu,1893.0,287933.0,23186.0,28125.0,27077.0,41077.0,15341.0,29416.0,37439.0,21308.0
Mombasa,6759.0,285669.5,30477.5,25144.5,17050.0,38344.0,47092.5,26730.0,16687.0,23624.0
Nairobi,7046.0,68967.0,52860.5,43107.5,41373.5,282931.0,56786.5,56432.5,65187.0,30934.5
Nakuru,2743.5,56574.0,28730.0,39795.0,18998.5,18145.0,276609.0,26152.5,30862.0,14875.0
Unknown,0.0,0.0,0.0,1639.0,0.0,8507.0,9824.0,0.0,1877.0,5309.0


### Task 17 Findings

Nairobi handled the highest total transaction value overall. Nairobi deposits generated KES 543,590 across 118 transactions, while withdrawals generated KES 162,036 across 62 transactions. Kisumu also recorded strong deposit activity, with KES 436,420.50 across 74 transactions. Nakuru had the highest average withdrawal value at KES 8,425.50, while Kiambu had the highest average deposit value at KES 7,784.58.

The weekly pivot table shows that transaction value varies considerably by county and week. Several large weekly totals are influenced by the KES 250,000 transactions that were flagged as outliers, so those weeks should be interpreted carefully. Transactions linked to unknown agents are retained under the `Unknown` county category so that valid transaction value is not lost from the analysis.

In [7]:
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

# Convert date to ISO format before export
export_df = clean_df.copy()
export_df["txn_date"] = export_df["txn_date"].dt.strftime("%Y-%m-%d")

export_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Created outputs/agent_transactions_clean.csv")

print("\nValidation:")
print("Rows:", len(export_df))
print("Duplicate txn_id values:", export_df["txn_id"].duplicated().sum())
print("Transaction types:", export_df["txn_type"].unique().tolist())
print("Amount dtype:", export_df["amount"].dtype)
print("Amount imputed flag present:", "amount_imputed" in export_df.columns)
print("Outlier flag present:", "is_outlier" in export_df.columns)
print("County column present:", "county" in export_df.columns)

display(export_df.head())

Created outputs/agent_transactions_clean.csv

Validation:
Rows: 600
Duplicate txn_id values: 0
Transaction types: ['Deposit', 'Withdrawal']
Amount dtype: float64
Amount imputed flag present: True
Outlier flag present: True
County column present: True


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded,week
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01,2026-03-02/2026-03-08
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01,2026-04-27/2026-05-03
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01,2026-04-27/2026-05-03
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01,2026-04-20/2026-04-26
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01,2026-04-13/2026-04-19


# One-Page Data Quality Report

## Completeness

The raw transaction file contained missing or invalid amount values. The initial audit found 22 affected raw rows, but after removing duplicate transaction IDs, 20 unique transactions required treatment. These amounts were converted to numeric values and the missing values were filled using the median transaction amount of KES 1,776.50. An `amount_imputed` flag was added so the affected records can still be identified. After cleaning, the transaction fields required for analysis had no missing values.

## Validity

Transaction amounts were converted to numeric format and transaction dates were converted to proper datetime values. After cleaning, no invalid transaction dates remained. Transaction types were also restricted to the two valid categories, `Deposit` and `Withdrawal`. The final cleaned dataset contains 373 Deposit transactions and 227 Withdrawal transactions. Five transactions with an amount of exactly KES 250,000 were identified as unusually high values and were flagged for investigation instead of being automatically removed.

## Accuracy

The agent IDs in the transaction file were checked against the agent register. The raw audit identified 8 transactions connected to unknown agents. After duplicate removal, 7 unique transactions remained with agent IDs that could not be matched to the register. A left join was used so these transactions were not lost, and their county was recorded as `Unknown`. These records should be checked against the source system to determine whether the agent IDs are incorrect or the agent register is incomplete.

## Consistency

The raw data contained inconsistent transaction type labels such as different capitalisation and extra spaces. A total of 384 raw rows did not already use the exact standard labels. These values were cleaned and standardised to `Deposit` and `Withdrawal`. Five raw transaction dates also used slash separators instead of the standard hyphen format. All transaction dates were converted to a consistent date format during cleaning.

## Uniqueness

The raw transaction dataset contained 612 rows and 12 duplicate transaction IDs. Duplicate records were removed using `txn_id` as the unique transaction key. After cleaning, the dataset contained 600 unique transactions and zero duplicated transaction IDs. This prevents duplicate transactions from incorrectly increasing totals, averages, and transaction counts.

## Timeliness

The transaction dates cover activity during March and April 2026, with the weekly summary extending from the week beginning February 23 through the week ending May 3 because weekly periods include the surrounding calendar dates. All cleaned transaction dates were successfully converted into a usable date format. The data is therefore suitable for the period represented in this analysis, although future business decisions should use the most recent transaction extract available.

## Outliers and Business Recommendation

The IQR method flagged 53 transaction amounts as possible outliers. Five of these were transactions worth exactly KES 250,000 and occurred across different agents, counties, dates and transaction types. Outliers were retained and flagged instead of deleted because an unusually high amount does not automatically mean that a transaction is incorrect. The business should verify high-value transactions against the original transaction system before correcting or excluding them.

## Privacy and Data Protection

Transaction histories and identifying information should be handled carefully because they may contain personal data. Under the Kenya Data Protection Act, 2019, organisations should use personal information only for a lawful purpose, collect only the information needed, maintain accurate records, and protect the data from unauthorised access. If customer names, phone numbers or other direct identifiers were included in this dataset, unnecessary fields should be removed and identifiers should be masked or pseudonymised before analysis or sharing. Raw customer information should not be uploaded to a public GitHub repository.

## Final Result

The cleaned dataset contains 600 unique transactions with standard transaction types, numeric amounts, ISO-formatted transaction dates, imputation and outlier flags, and agent county information. The cleaning process preserved valid transactions while documenting the issues that still require business review. The final cleaned file was exported as `outputs/agent_transactions_clean.csv`.